# 🧠 Target-Aware Query Cross-Attention & Component Interaction Pipeline
### End-to-End Compositionality Rating (1.0 → 5.0) for Noun Compounds (NC) & Particle Verbs (PV)

This notebook provides a complete, self-contained, and production-ready PyTorch implementation of the **Target-Aware Query Cross-Attention & Component Interaction Architecture**.

```
[INPUT SEQUENCE]
  - Noun Compound: "This was soon thrown out through the back door..." (Mod: back, Head: door)
  - Particle Verb: "Er haute sofort durch die Hintertür ab." (Verb: haute, Particle: ab)
                                      |
                                      v
+-------------------------------------------------------------------------------+
| LAYER 0: ENCODER & EXPLICIT ROLE INJECTION                                    |
|  Input Token IDs [B, S] ------> mmBERT Encoder ----> H_mmBERT  [B, S, 768]    |
|  Role IDs        [B, S] ------> Role Embeddings ---> E_role    [B, S, 768]    |
|  (0: Context, 1: Mod / Base Verb, 2: Head / Particle)                         |
|                                H_final = H_mmBERT + E_role ---> [B, S, 768]   |
+-------------------------------------------------------------------------------+
                                      |
                                      v
+-------------------------------------------------------------------------------+
| LAYER 1: TARGET-AWARE MULTI-HEAD CROSS-ATTENTION (MHCA)                       |
|  Learned Queries: Q_base = [q_Slot0, q_Slot1, q_Slot2] in [3, 768]             |
|  Probes H_final -> Softmax Attention Map A in [B, 3, S]                       |
|  Z = LayerNorm(Q + Dropout(MHCA(Q, K, V)))  --> Tensor Z in [B, 3, 768]       |
+-------------------------------------------------------------------------------+
                                      |
                                      v
+-------------------------------------------------------------------------------+
| LAYER 2: COMPONENT INTERACTION LAYER (MHSA)                                   |
|  Self-Attention between constituents (Mod/Verb <-> Head/Part <-> Whole Expr): |
|  Z' = LayerNorm(Z + Dropout(MHSA(Z, Z, Z))) --> Tensor Z' in [B, 3, 768]      |
+-------------------------------------------------------------------------------+
                                      |
                                      v
+-------------------------------------------------------------------------------+
| LAYER 3: SHARED GAUSSHEAD REGRESSION & CCC LOSS                               |
|  z'_Slot0, z'_Slot1, z'_Slot2 ----> Shared GaussHead MLP                     |
|  Outputs: (mu_0, sigma_0), (mu_1, sigma_1), (mu_2, sigma_2)                   |
|  Loss: Lin's Concordance Correlation Coefficient (CCC) + Gaussian KL Loss     |
|  Score Prediction: Continuous score = mu * 5.0 in [1.0, 5.0]                  |
+-------------------------------------------------------------------------------+
```

In [ ]:
# =============================================================================
# 1. Environment Setup & Dependency Verification
# =============================================================================
!pip install -q transformers scipy pandas numpy matplotlib scikit-learn

import os
import re
import math
import json
import random
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple, Sequence, Union, Any
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Running on device: {device} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
# 2. End-to-End Typed Configuration
# =============================================================================
@dataclass
class KagglePipelineConfig:
    seed: int = 42
    backbone: str = 'jhu-clsp/mmBERT-base' # Alternative: 'xlm-roberta-base' or 'bert-base-multilingual-cased'
    hidden_size: int = 768
    head_hidden: int = 128
    dropout: float = 0.1
    num_roles: int = 4            # 0: Context, 1: Mod / Base Verb, 2: Head / Particle, 3: Expr
    num_cross_heads: int = 8      # Layer 1 MHCA heads
    num_self_heads: int = 4       # Layer 2 MHSA heads
    shared_head: bool = True
    sigma_floor: float = 0.04
    
    # Data paths
    max_context_length: int = 256
    data_dir: str = '/kaggle/input/compositionality-dataset' # Or current workspace
    
    # Training parameters
    freeze_epochs: int = 3        # Phase 1: Train query base & heads only
    unfreeze_epochs: int = 6      # Phase 2: Unfreeze top encoder layers
    targets: List[str] = field(default_factory=lambda: ['mod', 'head', 'pv'])
    batch_size: int = 16
    eval_batch_size: int = 32
    lr_heads: float = 3e-4
    lr_backbone: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    fp16: bool = True
    
    # Loss settings
    ccc_weight: float = 0.7
    ccc_var_floor: float = 0.05
    bin_sigma: float = 0.5
    use_label_std: bool = True
    kl_weight: float = 1.0

cfg = KagglePipelineConfig()
print("📋 Pipeline Configuration:")
print(json.dumps(asdict(cfg), indent=2))

In [ ]:
# =============================================================================
# 3. Subtoken Span Finder & Discontinuous PV Alignment
# =============================================================================
_INFLECTION = (
    "'s", "\u2019s", "ses", "sen", "se", "es", "ed", "ing", "en", "ern", "er", "s", "e", "n",
)
_INFL_RE = "(?:" + "|".join(re.escape(s) for s in _INFLECTION) + ")?"
_FUGEN = r"(?:s|es|en|er|n|e|ens|-)?"
_LETTER = r"[^\W\d_]"
_START_BOUNDARY = f"(?<!{_LETTER})"
_END_BOUNDARY = f"(?!{_LETTER})"

_GERMAN_PARTICLES = (
    "ab", "an", "auf", "aus", "bei", "ein", "fest", "fort", "her", "hin",
    "los", "mit", "nach", "vor", "weg", "zu", "zurück", "zusammen",
)

_IRREGULAR_DE = {
    "hauen": ("hieb", "gehauen", "haut", "haust", "haute", "hauten"),
    "geben": ("gab", "gäbe", "gegeben", "gibt", "gibst"),
    "gehen": ("ging", "ginge", "gegangen", "geht", "gehst"),
    "kommen": ("kam", "käme", "gekommen", "kommt", "kommst"),
    "nehmen": ("nahm", "nähme", "genommen", "nimmt", "nimmst"),
    "räumen": ("geräumt", "räume", "räumt", "räumte", "abzuräumen"),
}

@dataclass(frozen=True)
class Span:
    start: Optional[int]
    end: Optional[int]

@dataclass(frozen=True)
class SpanResult:
    mod: Span
    head: Span
    degenerate: bool = False

def _char_span_to_token_span(offsets: Sequence[Tuple[int, int]], c_start: int, c_end: int) -> Span:
    tok_start: Optional[int] = None
    tok_end: Optional[int] = None
    for i, (s, e) in enumerate(offsets):
        if s == e: continue
        if e <= c_start: continue
        if s >= c_end: break
        if tok_start is None:
            tok_start = i
        tok_end = i + 1
    return Span(tok_start, tok_end)

def find_spans(context: str, mod: str, head: str, offsets: Sequence[Tuple[int, int]], lang: str = 'en') -> SpanResult:
    """Aligns mod/verb and head/particle in context without delimiter markers."""
    m_clean = mod.strip()
    h_clean = head.strip()
    if not m_clean or not h_clean:
        return SpanResult(Span(None, None), Span(None, None))

    # Strategy 1: Fused compounds ("Abiturzeugnis", "Liebesbrief")
    pat_fused = re.compile(f"{_START_BOUNDARY}({re.escape(m_clean)})({_FUGEN})({re.escape(h_clean)}{_INFL_RE}){_END_BOUNDARY}", re.IGNORECASE)
    m = pat_fused.search(context)
    if m:
        mod_span = _char_span_to_token_span(offsets, m.start(1), m.end(1))
        head_span = _char_span_to_token_span(offsets, m.start(3), m.end(3))
        return SpanResult(mod_span, head_span)

    # Strategy 2: Spaced compounds / adjacent PVs ("back door", "give up")
    pat_spaced = re.compile(f"{_START_BOUNDARY}({re.escape(m_clean)})\\s+({re.escape(h_clean)}{_INFL_RE}){_END_BOUNDARY}", re.IGNORECASE)
    m = pat_spaced.search(context)
    if m:
        mod_span = _char_span_to_token_span(offsets, m.start(1), m.end(1))
        head_span = _char_span_to_token_span(offsets, m.start(2), m.end(2))
        return SpanResult(mod_span, head_span)

    # Strategy 3: Independent / Discontinuous Search (e.g. "haute ... ab", "turned ... off")
    mod_forms = [m_clean]
    head_forms = [h_clean]
    if lang == 'de' and m_clean.lower() in _IRREGULAR_DE:
        mod_forms.extend(_IRREGULAR_DE[m_clean.lower()])
    if lang == 'de' and h_clean.lower() in _IRREGULAR_DE:
        head_forms.extend(_IRREGULAR_DE[h_clean.lower()])

    mod_re = re.compile(f"{_START_BOUNDARY}(" + "|".join(re.escape(f) for f in mod_forms) + f"{_INFL_RE}){_END_BOUNDARY}", re.IGNORECASE)
    head_re = re.compile(f"{_START_BOUNDARY}(" + "|".join(re.escape(f) for f in head_forms) + f"{_INFL_RE}){_END_BOUNDARY}", re.IGNORECASE)

    m_match = mod_re.search(context)
    h_match = head_re.search(context)
    if m_match and h_match:
        mod_span = _char_span_to_token_span(offsets, m_match.start(1), m_match.end(1))
        head_span = _char_span_to_token_span(offsets, h_match.start(1), h_match.end(1))
        return SpanResult(mod_span, head_span)

    return SpanResult(Span(None, None), Span(None, None))

In [ ]:
# =============================================================================
# 4. Dataset Loader, Target Expansion & Collate
# =============================================================================
class CompDataset(Dataset):
    """Prepares natural sequences with verbatim tokenization and explicit Role IDs."""
    def __init__(self, rows: List[Dict], tokenizer, max_len: int = 256):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        r = self.rows[idx]
        ctx = r['context']
        enc = self.tokenizer(
            ctx,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_offsets_mapping=True,
            return_tensors='pt'
        )
        input_ids = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)
        offsets = [tuple(o) for o in enc['offset_mapping'].squeeze(0).tolist()]

        spans = find_spans(ctx, r['mod'], r['head'], offsets, lang=r.get('lang', 'en'))

        # Role IDs: 0 = Context, 1 = Mod / Base Verb, 2 = Head / Particle
        role_ids = torch.zeros_like(input_ids)
        mod_mask = torch.zeros_like(input_ids, dtype=torch.bool)
        head_mask = torch.zeros_like(input_ids, dtype=torch.bool)

        if spans.mod.start is not None and spans.mod.end is not None:
            role_ids[spans.mod.start:spans.mod.end] = 1
            mod_mask[spans.mod.start:spans.mod.end] = True

        if spans.head.start is not None and spans.head.end is not None:
            role_ids[spans.head.start:spans.head.end] = 2
            head_mask[spans.head.start:spans.head.end] = True

        target_name = r.get('target', 'mod')
        if target_name == 'mod':
            label_mu = r.get('mod_avg', float('nan')) / 5.0
            label_std = r.get('mod_std', 0.5) / 5.0
            target_code = 0
        elif target_name == 'head':
            label_mu = r.get('head_avg', float('nan')) / 5.0
            label_std = r.get('head_std', 0.5) / 5.0
            target_code = 1
        else: # 'pv' or 'comp'
            label_mu = r.get('mod_avg', float('nan')) / 5.0
            label_std = r.get('mod_std', 0.5) / 5.0
            target_code = 2

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'role_ids': role_ids,
            'mod_span_mask': mod_mask,
            'head_span_mask': head_mask,
            'target_code': target_code,
            'label_mu': torch.tensor(label_mu, dtype=torch.float32),
            'label_std': torch.tensor(label_std, dtype=torch.float32),
            'has_label': torch.tensor(np.isfinite(label_mu), dtype=torch.bool),
            'context': ctx,
            'mod': r['mod'],
            'head': r['head'],
        }

def collate_comp(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    return {
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'role_ids': torch.stack([b['role_ids'] for b in batch]),
        'mod_span_mask': torch.stack([b['mod_span_mask'] for b in batch]),
        'head_span_mask': torch.stack([b['head_span_mask'] for b in batch]),
        'target_code': torch.tensor([b['target_code'] for b in batch], dtype=torch.long),
        'label_mu': torch.stack([b['label_mu'] for b in batch]),
        'label_std': torch.stack([b['label_std'] for b in batch]),
        'has_label': torch.stack([b['has_label'] for b in batch]),
    }

def expand_targets(rows: List[Dict], targets: List[str]) -> List[Dict]:
    """Expands each multi-target row into single-target supervised rows."""
    out = []
    for r in rows:
        is_pv = r.get('is_pv', False)
        for t in targets:
            if (not is_pv and t in ('mod', 'head')) or (is_pv and t == 'pv'):
                c = dict(r)
                c['target'] = t
                out.append(c)
    return out

In [ ]:
# =============================================================================
# 5. Target-Aware Neural Architecture (Layers 0 -> 3)
# =============================================================================
class GaussHead(nn.Module):
    """Layer 3: Shared Gaussian Regression Head outputting (mu, sigma)."""
    def __init__(self, in_features: int = 768, hidden_dim: int = 128, dropout: float = 0.1, sigma_floor: float = 0.04):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )
        self.sigma_floor = sigma_floor

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        out = self.mlp(x)
        mu = torch.sigmoid(out[..., 0]) # Normalized mu in [0, 1]
        sigma = F.softplus(out[..., 1]) + self.sigma_floor # Spread variance > 0
        return mu, sigma

class TargetAwareCrossAttention(nn.Module):
    """Layer 1: MHCA with learned queries [q_Slot0, q_Slot1, q_Slot2] probing H_final."""
    def __init__(self, hidden_size: int = 768, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.q_base = nn.Parameter(torch.empty(3, hidden_size))
        nn.init.normal_(self.q_base, std=0.02)
        
        self.mha = nn.MultiheadAttention(embed_dim=hidden_size, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h_final: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        B = h_final.size(0)
        q = self.q_base.unsqueeze(0).expand(B, -1, -1) # [B, 3, 768]
        attn_out, attn_weights = self.mha(query=q, key=h_final, value=h_final, key_padding_mask=key_padding_mask)
        z = self.norm(q + self.dropout(attn_out)) # [B, 3, 768]
        return z, attn_weights

class ComponentInteractionLayer(nn.Module):
    """Layer 2: MHSA modeling interactions between constituents (Mod/Verb <-> Head/Part <-> Expr)."""
    def __init__(self, hidden_size: int = 768, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=hidden_size, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, z: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        attn_out, attn_weights = self.self_attn(query=z, key=z, value=z)
        z_prime = self.norm(z + self.dropout(attn_out)) # [B, 3, 768]
        return z_prime, attn_weights

class TargetAwareQueryAttentionModel(nn.Module):
    """Unified End-to-End Compositionality Model (Layers 0 -> 3)."""
    def __init__(self, cfg: KagglePipelineConfig):
        super().__init__()
        self.cfg = cfg
        self.lm = AutoModel.from_pretrained(cfg.backbone)
        self.role_embed = nn.Embedding(cfg.num_roles, cfg.hidden_size)
        
        self.cross_attn = TargetAwareCrossAttention(hidden_size=cfg.hidden_size, num_heads=cfg.num_cross_heads, dropout=cfg.dropout)
        self.comp_inter = ComponentInteractionLayer(hidden_size=cfg.hidden_size, num_heads=cfg.num_self_heads, dropout=cfg.dropout)
        self.gauss_head = GaussHead(in_features=cfg.hidden_size, hidden_dim=cfg.head_hidden, dropout=cfg.dropout, sigma_floor=cfg.sigma_floor)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, role_ids: torch.Tensor) -> Dict[str, torch.Tensor]:
        # Layer 0: Encoder & Explicit Role Injection
        lm_out = self.lm(input_ids=input_ids, attention_mask=attention_mask)
        h_final = lm_out.last_hidden_state + self.role_embed(role_ids) # [B, S, 768]
        
        # Layer 1: Target-Aware MHCA
        key_padding_mask = (attention_mask == 0)
        z, cross_attn_map = self.cross_attn(h_final, key_padding_mask=key_padding_mask) # [B, 3, 768]
        
        # Layer 2: Component Interaction MHSA
        z_prime, comp_matrix = self.comp_inter(z) # [B, 3, 768]
        
        # Layer 3: Shared GaussHead
        mu_all, sigma_all = self.gauss_head(z_prime) # [B, 3], [B, 3]
        
        return {
            'mu_0': mu_all[:, 0],
            'sigma_0': sigma_all[:, 0],
            'mu_1': mu_all[:, 1],
            'sigma_1': sigma_all[:, 1],
            'mu_2': mu_all[:, 2],
            'sigma_2': sigma_all[:, 2],
            'mu_all': mu_all,
            'sigma_all': sigma_all,
            'cross_attn_map': cross_attn_map,
            'comp_matrix': comp_matrix,
        }

    def pred_heads(self) -> List[nn.Module]:
        return [self.role_embed, self.cross_attn, self.comp_inter, self.gauss_head]

In [ ]:
# =============================================================================
# 6. Loss Functions: Lin's CCC Loss + Gaussian KL Divergence
# =============================================================================
def ccc_loss(pred: torch.Tensor, target: torch.Tensor, var_floor: float = 0.05, eps: float = 1e-8) -> torch.Tensor:
    """Lin's Concordance Correlation Coefficient Loss: 1 - CCC(pred, target)."""
    if pred.numel() < 2:
        return F.mse_loss(pred, target)
    
    mu_x = pred.mean()
    mu_y = target.mean()
    var_x = ((pred - mu_x) ** 2).mean()
    var_y = ((target - mu_y) ** 2).mean()
    var_x_safe = torch.clamp(var_x, min=var_floor)
    var_y_safe = torch.clamp(var_y, min=var_floor)
    
    cov_xy = ((pred - mu_x) * (target - mu_y)).mean()
    ccc = (2.0 * cov_xy) / (var_x_safe + var_y_safe + (mu_x - mu_y) ** 2 + eps)
    return 1.0 - ccc

class GaussLoss(nn.Module):
    """Combined Lin's CCC Loss and Gaussian KL Loss."""
    def __init__(self, ccc_weight: float = 0.7, ccc_var_floor: float = 0.05, bin_sigma: float = 0.5, use_label_std: bool = True, kl_weight: float = 1.0):
        super().__init__()
        self.ccc_weight = ccc_weight
        self.ccc_var_floor = ccc_var_floor
        self.bin_sigma = bin_sigma
        self.use_label_std = use_label_std
        self.kl_weight = kl_weight

    def forward(self, mu: torch.Tensor, sigma: torch.Tensor, y_mu: torch.Tensor, y_std: torch.Tensor) -> torch.Tensor:
        t_sigma = (y_std if self.use_label_std else torch.full_like(y_std, self.bin_sigma)).clamp(min=1e-4)
        
        # KL Divergence: KL(N(mu, sigma^2) || N(y_mu, t_sigma^2))
        var_p = sigma ** 2
        var_t = t_sigma ** 2
        kl = torch.log(t_sigma / sigma) + (var_p + (mu - y_mu) ** 2) / (2.0 * var_t) - 0.5
        loss_kl = kl.mean()
        
        loss_ccc = ccc_loss(mu, y_mu, var_floor=self.ccc_var_floor)
        return (1.0 - self.ccc_weight) * loss_kl + self.ccc_weight * loss_ccc

In [ ]:
# =============================================================================
# 7. 2-Phase Training Loop & Validation Metrics
# =============================================================================
def unfreeze_top_layers(model: nn.Module, num_layers: int = 4):
    """Selectively unfreezes the top N transformer encoder layers."""
    encoder = getattr(model.lm, 'encoder', None)
    if encoder is not None and hasattr(encoder, 'layer'):
        layers = encoder.layer
    elif hasattr(model.lm, 'transformer') and hasattr(model.lm.transformer, 'layer'):
        layers = model.lm.transformer.layer
    else:
        for param in model.lm.parameters():
            param.requires_grad = True
        return

    for layer in layers[-num_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

def train_epoch(model, loader, optimizer, criterion, scheduler, scaler, device) -> float:
    model.train()
    total_loss = 0.0
    count = 0
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        role_ids = batch['role_ids'].to(device)
        target_code = batch['target_code'].to(device)
        label_mu = batch['label_mu'].to(device)
        label_std = batch['label_std'].to(device)
        has_label = batch['has_label'].to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda', enabled=scaler is not None):
            out = model(input_ids=input_ids, attention_mask=attention_mask, role_ids=role_ids)
            mu_all = out['mu_all']       # [B, 3]
            sigma_all = out['sigma_all'] # [B, 3]
            
            # Route active target per sample
            batch_indices = torch.arange(input_ids.size(0), device=device)
            mu_active = mu_all[batch_indices, target_code]
            sigma_active = sigma_all[batch_indices, target_code]
            
            valid = has_label & torch.isfinite(label_mu)
            if not valid.any():
                continue
                
            loss = criterion(mu_active[valid], sigma_active[valid], label_mu[valid], label_std[valid])
        
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
        scheduler.step()
        total_loss += loss.item()
        count += 1
        
    return total_loss / max(count, 1)

@torch.no_grad()
def evaluate(model, loader, device) -> Dict[str, float]:
    model.eval()
    preds, labels, targets = [], [], []
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        role_ids = batch['role_ids'].to(device)
        target_code = batch['target_code'].to(device)
        label_mu = batch['label_mu'].to(device)
        
        out = model(input_ids=input_ids, attention_mask=attention_mask, role_ids=role_ids)
        batch_indices = torch.arange(input_ids.size(0), device=device)
        mu_active = out['mu_all'][batch_indices, target_code] * 5.0 # Un-normalize to [1, 5]
        
        preds.extend(mu_active.cpu().numpy())
        labels.extend((label_mu * 5.0).cpu().numpy())
        targets.extend(target_code.cpu().numpy())
        
    preds = np.array(preds)
    labels = np.array(labels)
    targets = np.array(targets)
    
    valid = np.isfinite(labels)
    preds, labels, targets = preds[valid], labels[valid], targets[valid]
    
    def calc_rho(mask):
        if mask.sum() < 3: return 0.0
        r, _ = spearmanr(preds[mask], labels[mask])
        return float(r) if np.isfinite(r) else 0.0
        
    rho_mod = calc_rho(targets == 0)
    rho_head = calc_rho(targets == 1)
    rho_pv = calc_rho(targets == 2)
    rho_mean = np.mean([r for r in [rho_mod, rho_head, rho_pv] if r > 0.0] or [0.0])
    
    return {'rho_mod': rho_mod, 'rho_head': rho_head, 'rho_pv': rho_pv, 'rho_mean': float(rho_mean)}

In [ ]:
# =============================================================================
# 8. Dataset Instantiation (Built-in Benchmark or Kaggle TSV Files)
# =============================================================================
SAMPLE_BENCHMARK = [
    # Noun Compounds (En & De)
    {'context': 'We spent the sunny Saturday morning browsing vintage records and lamps at the local flea market.', 'mod': 'flea', 'head': 'market', 'compound': 'flea market', 'lang': 'en', 'is_pv': False, 'mod_avg': 1.4, 'head_avg': 4.8, 'mod_std': 0.45, 'head_std': 0.35},
    {'context': 'The legendary actress made her triumphant return to the silver screen after a decade in theater.', 'mod': 'silver', 'head': 'screen', 'compound': 'silver screen', 'lang': 'en', 'is_pv': False, 'mod_avg': 1.15, 'head_avg': 1.8, 'mod_std': 0.62, 'head_std': 0.55},
    {'context': 'She packed two ripe apples and a cheese sandwich into a brown paper bag for lunch.', 'mod': 'paper', 'head': 'bag', 'compound': 'paper bag', 'lang': 'en', 'is_pv': False, 'mod_avg': 4.95, 'head_avg': 4.95, 'mod_std': 0.18, 'head_std': 0.18},
    {'context': 'Developing nations frequently suffer from brain drain when top researchers seek opportunities abroad.', 'mod': 'brain', 'head': 'drain', 'compound': 'brain drain', 'lang': 'en', 'is_pv': False, 'mod_avg': 2.10, 'head_avg': 2.50, 'mod_std': 0.85, 'head_std': 0.75},
    {'context': 'Nach den anstrengenden Prüfungen hielt sie stolz ihr frisch gedrucktes Abiturzeugnis in den Händen.', 'mod': 'Abitur', 'head': 'Zeugnis', 'compound': 'Abiturzeugnis', 'lang': 'de', 'is_pv': False, 'mod_avg': 4.88, 'head_avg': 4.88, 'mod_std': 0.28, 'head_std': 0.28},
    {'context': 'In der alten Schatulle fand er einen vergilbten Liebesbrief aus dem Jahr 1942.', 'mod': 'Liebe', 'head': 'Brief', 'compound': 'Liebesbrief', 'lang': 'de', 'is_pv': False, 'mod_avg': 4.75, 'head_avg': 4.75, 'mod_std': 0.32, 'head_std': 0.32},
    
    # Particle Verbs (En & De - Continuous & Discontinuous)
    {'context': 'Even after three grueling attempts, the mountaineer refused to give up on reaching the summit.', 'mod': 'give', 'head': 'up', 'compound': 'give up', 'lang': 'en', 'is_pv': True, 'mod_avg': 0.85, 'head_avg': 0.85, 'mod_std': 0.58, 'head_std': 0.58},
    {'context': 'Before leaving the office, please remember to turn all the conference room lights off.', 'mod': 'turn', 'head': 'off', 'compound': 'turn off', 'lang': 'en', 'is_pv': True, 'mod_avg': 3.80, 'head_avg': 3.80, 'mod_std': 0.40, 'head_std': 0.40},
    {'context': 'Die Kellnerin begann zügig die leeren Teller vom Tisch abzuräumen.', 'mod': 'ab', 'head': 'räumen', 'compound': 'abräumen', 'lang': 'de', 'is_pv': True, 'mod_avg': 4.10, 'head_avg': 4.10, 'mod_std': 0.42, 'head_std': 0.42},
    {'context': 'Als die Polizei eintraf, haute der Dieb sofort durch die Hintertür ab.', 'mod': 'haute', 'head': 'ab', 'compound': 'abhauen', 'lang': 'de', 'is_pv': True, 'mod_avg': 0.65, 'head_avg': 0.65, 'mod_std': 0.70, 'head_std': 0.70},
]

print(f"📦 Loaded {len(SAMPLE_BENCHMARK)} sample benchmark records.")
tokenizer = AutoTokenizer.from_pretrained(cfg.backbone)

# Expand active target rows
train_rows = expand_targets(SAMPLE_BENCHMARK, cfg.targets)
val_rows = expand_targets(SAMPLE_BENCHMARK, cfg.targets)

train_loader = DataLoader(CompDataset(train_rows, tokenizer, max_len=cfg.max_context_length), batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_comp)
val_loader = DataLoader(CompDataset(val_rows, tokenizer, max_len=cfg.max_context_length), batch_size=cfg.eval_batch_size, shuffle=False, collate_fn=collate_comp)

print(f"🔹 Train Samples (Target Expanded): {len(train_rows)} | Val Samples: {len(val_rows)}")

In [ ]:
# =============================================================================
# 9. Full End-to-End Training Execution
# =============================================================================
model = TargetAwareQueryAttentionModel(cfg).to(device)
criterion = GaussLoss(
    ccc_weight=cfg.ccc_weight, 
    ccc_var_floor=cfg.ccc_var_floor, 
    bin_sigma=cfg.bin_sigma, 
    use_label_std=cfg.use_label_std, 
    kl_weight=cfg.kl_weight
).to(device)

scaler = torch.amp.GradScaler('cuda') if (cfg.fp16 and torch.cuda.is_available()) else None

# -----------------------------------------------------------------------------
# Phase 1: Freeze Backbone, Train Query MHCA & GaussHeads
# -----------------------------------------------------------------------------
print("\n--- 🛡️ Phase 1: Training Query Attention & Heads (Backbone Frozen) ---")
for param in model.lm.parameters():
    param.requires_grad = False

head_params = [p for m in model.pred_heads() for p in m.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(head_params, lr=cfg.lr_heads, weight_decay=cfg.weight_decay)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(len(train_loader)*cfg.freeze_epochs*cfg.warmup_ratio), num_training_steps=len(train_loader)*cfg.freeze_epochs)

for epoch in range(1, cfg.freeze_epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, scheduler, scaler, device)
    metrics = evaluate(model, val_loader, device)
    print(f"Epoch {epoch:02d}/{cfg.freeze_epochs:02d} | Train Loss: {loss:.4f} | Val Rho Mean: {metrics['rho_mean']:.4f} (Mod: {metrics['rho_mod']:.4f}, Head: {metrics['rho_head']:.4f}, PV: {metrics['rho_pv']:.4f})")

# -----------------------------------------------------------------------------
# Phase 2: Unfreeze Top Layers with Discriminative Learning Rate
# -----------------------------------------------------------------------------
print("\n--- 🔓 Phase 2: Fine-Tuning Top Encoder Layers & Heads ---")
unfreeze_top_layers(model, num_layers=4)

backbone_params = [p for p in model.lm.parameters() if p.requires_grad]
head_params = [p for m in model.pred_heads() for p in m.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg.lr_backbone},
    {'params': head_params, 'lr': cfg.lr_heads}
], weight_decay=cfg.weight_decay)

p2_steps = len(train_loader) * cfg.unfreeze_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(p2_steps * cfg.warmup_ratio), num_training_steps=p2_steps)

for epoch in range(cfg.freeze_epochs + 1, cfg.freeze_epochs + cfg.unfreeze_epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, scheduler, scaler, device)
    metrics = evaluate(model, val_loader, device)
    print(f"Epoch {epoch:02d}/{(cfg.freeze_epochs + cfg.unfreeze_epochs):02d} | Train Loss: {loss:.4f} | Val Rho Mean: {metrics['rho_mean']:.4f} (Mod: {metrics['rho_mod']:.4f}, Head: {metrics['rho_head']:.4f}, PV: {metrics['rho_pv']:.4f})")

# Save trained checkpoint
torch.save(model.state_dict(), 'target_aware_model.pt')
print("\n✅ Checkpoint saved: target_aware_model.pt")

In [ ]:
# =============================================================================
# 10. Interactive Single-Sample Inference & Attention Heatmap Visualizer
# =============================================================================
@torch.no_grad()
def predict_sample(sentence: str, mod: str, head: str, lang: str = 'en'):
    model.eval()
    enc = tokenizer(sentence, return_offsets_mapping=True, return_tensors='pt', max_length=128, padding='max_length', truncation=True)
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    offsets = [tuple(o) for o in enc['offset_mapping'].squeeze(0).tolist()]
    
    spans = find_spans(sentence, mod, head, offsets, lang=lang)
    role_ids = torch.zeros_like(input_ids)
    if spans.mod.start is not None: role_ids[0, spans.mod.start:spans.mod.end] = 1
    if spans.head.start is not None: role_ids[0, spans.head.start:spans.head.end] = 2
    
    out = model(input_ids=input_ids, attention_mask=attention_mask, role_ids=role_ids)
    
    mu_0 = out['mu_0'].item() * 5.0
    sigma_0 = out['sigma_0'].item() * 5.0
    mu_1 = out['mu_1'].item() * 5.0
    sigma_1 = out['sigma_1'].item() * 5.0
    mu_2 = out['mu_2'].item() * 5.0
    sigma_2 = out['sigma_2'].item() * 5.0
    
    print(f"\n📝 Sentence: '{sentence}'")
    print(f"🎯 Target Metadata: Slot 0 ('{mod}'), Slot 1 ('{head}')")
    print(f"📊 Continuous Compositionality Rating Predictions (1.0 -> 5.0):")
    print(f"  • Slot 0 (Mod / Base Verb):   Score = {mu_0:.2f} (± {sigma_0:.2f})")
    print(f"  • Slot 1 (Head / Particle):    Score = {mu_1:.2f} (± {sigma_1:.2f})")
    print(f"  • Slot 2 (Whole Expression):   Score = {mu_2:.2f} (± {sigma_2:.2f})")

# Test on Noun Compound & Particle Verb cases
predict_sample("We bought an antique table at the local flea market.", "flea", "market", "en")
predict_sample("Er haute sofort durch die Hintertür ab.", "haute", "ab", "de")